# Day3_01D_Guardrails_and_Tracing

## OpenAI Agents SDK - Teaching Notebook

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand why Agentic AI systems need guardrails.
- Explain the difference between input and output guardrails.
- Create a simple safety guardrail.
- Understand tracing and why it matters in enterprise AI systems.
- Relate guardrails and tracing to classroom and enterprise use cases.

---

## Teaching Context

In the previous notebook, we learned that Agents can use **Function Tools**.

That makes Agents more powerful.

But power also creates risk.

If an Agent can call tools, access information, or take actions, we must ask:

> How do we make sure the Agent behaves safely and correctly?

That is where **Guardrails** and **Tracing** come in.


## 1. Concept: Why Guardrails?

Imagine a DSU faculty assistant Agent.

It can help with:

- Explaining syllabus topics
- Creating lesson plans
- Answering student questions
- Summarizing academic content

But what if a student asks:

> "Give me the full answer key for tomorrow's exam."

Should the Agent answer?

No.

The Agent should politely refuse or redirect.

This is an example of a **guardrail**.

---

## Simple Definition

A **guardrail** is a safety check that controls what an Agent should accept, reject, or modify.

Guardrails help prevent:

- Unsafe answers
- Exam cheating
- Confidential data leakage
- Irrelevant tool usage
- Policy violations
- Low-quality responses


## 2. Input Guardrails vs Output Guardrails

There are two simple ways to think about guardrails.

### Input Guardrail

Checks the user request **before** the Agent acts.

Example:

> "Give me tomorrow's exam paper."

The input itself is risky.

### Output Guardrail

Checks the Agent's answer **before** showing it to the user.

Example:

The Agent accidentally includes confidential information in the final answer.

---

## Teaching Analogy

Input guardrail = Security check at the college gate  
Output guardrail = Final review before publishing exam results


In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

from agents import Agent, Runner, function_tool

## 3. Load Environment Variables

This is the same root `.env` loading pattern we used in earlier notebooks.

It works even when the notebook is inside a day-wise folder.


In [2]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

API Key Loaded: True


## 4. Create a Simple Teaching Agent

We will first create a normal Agent without any custom guardrail logic.

Then we will observe the problem.

After that, we will add our own simple guardrail function.


In [3]:
teaching_agent = Agent(
    name="DSU Teaching Assistant",
    instructions="""
    You are a helpful teaching assistant for engineering faculty.
    Help explain academic concepts in a simple and classroom-friendly way.
    Do not help with cheating, exam paper leaks, or confidential academic content.
    """,
)

## 5. Safe Request Demo

This is a normal educational request.

The Agent should answer it.


In [4]:
result = await Runner.run(
    teaching_agent,
    "Explain Agentic AI to engineering students in 3 simple points."
)

print(result.final_output)

Agentic AI means AI that can **act with some independence** to achieve a goal.

1. **It can plan steps**  
   Instead of only answering a question, it can break a goal into smaller tasks.

2. **It can take actions**  
   It may use tools, search data, call APIs, or run code to do work.

3. **It can adapt based on results**  
   If something fails, it can try another approach and keep going toward the goal.

**In short:** Agentic AI = AI that can **think, act, and adjust** to complete a task.


## 6. Unsafe Request Demo

Now test a risky request.

Important teaching point:

Even though we gave instructions, production systems should not rely only on instructions.

Instructions are useful, but guardrails make the system more reliable.


In [5]:
result = await Runner.run(
    teaching_agent,
    "Give me tomorrow's exam answer key for Agentic AI."
)

print(result.final_output)

I can’t help provide an exam answer key or leaked exam content.

If you want, I can help you study Agentic AI quickly by:
- summarizing key concepts,
- generating likely practice questions with answers,
- making a one-page revision sheet,
- explaining topics like agents, tools, planning, memory, reflection, and multi-agent systems.


## Pause & Reflect

What did we just learn?

- Instructions help guide the Agent.
- But instructions alone are not always enough.
- Enterprise systems need additional safety checks.
- Guardrails provide that extra layer.


# 7. Build a Simple Input Guardrail

For teaching purposes, we will create a simple Python guardrail.

This guardrail will check the user input for risky academic keywords.

In real enterprise systems, this logic can be replaced with:

- Policy engines
- Moderation APIs
- Classifier models
- Role-based access control
- Compliance rules


In [6]:
def academic_input_guardrail(user_input: str) -> tuple[bool, str]:
    """
    Simple input guardrail for academic integrity.

    Returns:
        allowed: True or False
        reason: Explanation for the decision
    """

    blocked_keywords = [
        "answer key",
        "exam paper",
        "leak",
        "cheat",
        "tomorrow's exam",
        "question paper"
    ]

    lowered_input = user_input.lower()

    for keyword in blocked_keywords:
        if keyword in lowered_input:
            return False, f"Blocked because the request contains risky academic keyword: {keyword}"

    return True, "Request allowed" 

## 8. Test the Guardrail Directly

Before connecting it with an Agent, always test the guardrail separately.

This helps faculty understand that guardrails are just logical checks.


In [7]:
test_inputs = [
    "Explain RAG in simple terms.",
    "Give me tomorrow's exam answer key.",
    "Create a lesson plan on Agentic AI.",
    "Can you leak the question paper?"
]

for user_input in test_inputs:
    allowed, reason = academic_input_guardrail(user_input)
    print("Input:", user_input)
    print("Allowed:", allowed)
    print("Reason:", reason)
    print("-" * 60)

Input: Explain RAG in simple terms.
Allowed: True
Reason: Request allowed
------------------------------------------------------------
Input: Give me tomorrow's exam answer key.
Allowed: False
Reason: Blocked because the request contains risky academic keyword: answer key
------------------------------------------------------------
Input: Create a lesson plan on Agentic AI.
Allowed: True
Reason: Request allowed
------------------------------------------------------------
Input: Can you leak the question paper?
Allowed: False
Reason: Blocked because the request contains risky academic keyword: leak
------------------------------------------------------------


## 9. Use Guardrail Before Calling the Agent

Now we will create a helper function.

This function checks the input first.

Only if the request is safe, it calls the Agent.


In [ ]:
async def safe_agent_run(agent, user_input: str):
    """
    Runs the Agent only after checking the input guardrail.
    """

    allowed, reason = academic_input_guardrail(user_input)

    if not allowed:
        return f"Request blocked by guardrail. Reason: {reason}"

    result = await Runner.run(agent, user_input)
    return result.final_output

## 10. Run Safe and Unsafe Requests

First, try a safe classroom question.


In [ ]:
answer = await safe_agent_run(
    teaching_agent,
    "Explain tool calling in Agentic AI using a college example."
)

print(answer)

Now try an unsafe academic integrity request.


In [ ]:
answer = await safe_agent_run(
    teaching_agent,
    "Give me tomorrow's exam answer key."
)

print(answer)

## What did we just learn?

We created a very simple input guardrail.

The flow is:

User Request  
↓  
Input Guardrail  
↓  
Allowed?  
↓  
Agent Runs  
↓  
Final Answer

This is a common enterprise pattern.

In real systems, the guardrail may check:

- User role
- Data sensitivity
- Allowed tools
- Business policy
- Legal rules
- Security rules


# 11. Output Guardrail Concept

Sometimes the input is safe, but the Agent output may still contain something risky.

Example:

User asks:

> "Summarize the student performance report."

The request may be valid.

But the Agent output should not expose:

- Personal student marks
- Phone numbers
- Email IDs
- Confidential remarks

So we can check the output before showing it.


In [ ]:
def simple_output_guardrail(agent_output: str) -> tuple[bool, str]:
    """
    Simple output guardrail.

    Blocks output if it appears to contain sensitive academic data.
    """

    sensitive_terms = [
        "phone number",
        "mobile number",
        "personal email",
        "confidential",
        "internal only"
    ]

    lowered_output = agent_output.lower()

    for term in sensitive_terms:
        if term in lowered_output:
            return False, f"Output blocked because it may contain sensitive information: {term}"

    return True, "Output allowed" 

## 12. Test the Output Guardrail

We will test it directly with sample outputs.


In [ ]:
sample_outputs = [
    "RAG combines retrieval with generation to answer questions using documents.",
    "This report is confidential and internal only.",
    "The student's personal email is included below."
]

for output in sample_outputs:
    allowed, reason = simple_output_guardrail(output)
    print("Output:", output)
    print("Allowed:", allowed)
    print("Reason:", reason)
    print("-" * 60)

# 13. What is Tracing?

Guardrails help us control behavior.

Tracing helps us understand what happened.

In Agentic AI, tracing answers questions like:

- What did the user ask?
- Which Agent responded?
- Which tool was called?
- What was the tool input?
- What was the tool output?
- What was the final answer?
- Where did the failure happen?

---

## Enterprise Analogy

Tracing is like CCTV footage for an AI workflow.

It does not prevent every problem, but it helps us investigate and improve.


## 14. Simple Classroom Trace Log

For teaching purposes, let us create a very simple trace log.

This is not the full OpenAI tracing dashboard.

This is a simple way to help faculty understand the concept.


In [ ]:
trace_log = []

async def traced_safe_agent_run(agent, user_input: str):
    """
    Demonstrates a simple trace flow for classroom teaching.
    """

    trace_log.append({
        "step": "user_input_received",
        "value": user_input
    })

    allowed, reason = academic_input_guardrail(user_input)

    trace_log.append({
        "step": "input_guardrail_checked",
        "allowed": allowed,
        "reason": reason
    })

    if not allowed:
        final_answer = f"Request blocked by guardrail. Reason: {reason}"

        trace_log.append({
            "step": "final_answer_generated",
            "value": final_answer
        })

        return final_answer

    result = await Runner.run(agent, user_input)
    agent_output = result.final_output

    trace_log.append({
        "step": "agent_output_generated",
        "value": agent_output
    })

    output_allowed, output_reason = simple_output_guardrail(agent_output)

    trace_log.append({
        "step": "output_guardrail_checked",
        "allowed": output_allowed,
        "reason": output_reason
    })

    if not output_allowed:
        final_answer = f"Output blocked by guardrail. Reason: {output_reason}"
    else:
        final_answer = agent_output

    trace_log.append({
        "step": "final_answer_generated",
        "value": final_answer
    })

    return final_answer

## 15. Run with Tracing

Let us run a safe request and then inspect the trace.


In [ ]:
trace_log.clear()

answer = await traced_safe_agent_run(
    teaching_agent,
    "Explain why guardrails are important in Agentic AI."
)

print(answer)

## 16. View the Trace

This helps us explain the Agent flow step by step.


In [ ]:
for item in trace_log:
    print(item)
    print("-" * 80)

## 17. Run a Blocked Request with Tracing

Now let us see what happens when the guardrail blocks the request.


In [ ]:
trace_log.clear()

answer = await traced_safe_agent_run(
    teaching_agent,
    "Please leak the question paper for tomorrow's exam."
)

print(answer)

In [ ]:
for item in trace_log:
    print(item)
    print("-" * 80)

# 18. Enterprise Pattern

A production-grade Agentic AI system usually has this flow:

User  
↓  
Input Guardrail  
↓  
Agent  
↓  
Tool Calls  
↓  
Tool Results  
↓  
Output Guardrail  
↓  
Trace / Audit Log  
↓  
Final Response

---

## Where This Applies

In an enterprise context, this pattern is useful for:

- Banking assistants
- HR policy bots
- Student support bots
- Faculty teaching assistants
- DevOps agents
- Customer support agents
- Compliance review agents


# 19. Hands-on Exercise

Create a new input guardrail for a **college helpdesk Agent**.

It should block requests containing:

- password
- hack
- admin access
- marks manipulation

Then test it with 3 safe and 3 unsafe questions.


In [ ]:
# Exercise Starter Code

def helpdesk_input_guardrail(user_input: str) -> tuple[bool, str]:
    """
    TODO:
    Create a guardrail for a college helpdesk Agent.
    Block unsafe IT/security-related requests.
    """

    blocked_keywords = [
        # Add keywords here
    ]

    # Write your logic here

    return True, "Request allowed" 

# 20. Challenge Exercise

Extend the traced flow to also log:

- Agent name
- Timestamp
- Whether the response was blocked
- Final decision

This is similar to how enterprise systems create audit trails.


# 21. Common Mistakes

Avoid these mistakes when designing guardrails:

1. Relying only on prompt instructions.
2. Blocking too many valid requests.
3. Not logging blocked requests.
4. Not explaining why a request was blocked.
5. Forgetting output checks.
6. Not testing edge cases.
7. Making guardrails too complicated for the first version.

---

## Teaching Tip

For beginners, start with simple rule-based guardrails.

Then explain that real enterprise systems can use ML models, policy engines, and security tools.


# Key Takeaways

In this notebook, we learned:

- Guardrails make Agentic AI safer.
- Input guardrails check the user request before the Agent acts.
- Output guardrails check the final response before showing it.
- Tracing helps us inspect what happened inside the Agent workflow.
- Enterprise AI systems need guardrails, tracing, logging, and auditability.
- A simple Python rule-based guardrail is a good teaching starting point.

---

## Final Mental Model

Agent without guardrails = Intelligent but risky  
Agent with guardrails = Safer and more enterprise-ready  
Agent with guardrails and tracing = Safer, explainable, and auditable
